# 第 16 天：顺序很重要——指针网络

**论文：**《顺序很重要：面向集合的序列到序列学习》（*Order Matters: Sequence to Sequence for Sets*）——Vinyals、Bengio、Kudlur（2015）

本 Notebook 实现指针网络（Pointer Network）：它通过选择输入集合中的索引生成输出序列，而非从固定词表生成词元。内容包括指针注意力、排序任务，以及一个小型 TSP 演示。

---

## 你将学到什么

1. 为什么输出词表依赖输入时，标准 seq2seq 方法会遇到困难
2. 指针注意力机制（原仓库标注为论文公式 1-3，编号待一手核验）
3. 读取—处理—写出框架如何处理集合结构的输入
4. 将排序用作指针网络的最小测试任务
5. 将指针网络用于旅行商问题

> **译者核验注：**这是仓库的教学实现，并非论文实验的完整复现。此处编码器使用 LSTM，仍对输入顺序敏感；没有实现独立的多步 PROCESS 模块。后文排序模型先产生 softmax 概率，再将其作为 `cross_entropy` 的输入，并在热力图处再次执行 softmax；这些是原有实现问题，本次仅记录，未修改算法。论文实验与章节编号尚待一手来源核验。

In [ ]:
# 初始化
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# 设置随机种子，便于复现
torch.manual_seed(42)
np.random.seed(42)

print('初始化完成')
print(f"PyTorch 版本：{torch.__version__}")
print(f"设备：{'GPU' if torch.cuda.is_available() else 'CPU'}")

## 第 1 部分：理解指针注意力

核心创新是指针机制。模型不从固定词表选择词元，而是在输入元素上计算注意力，并根据得到的分布选择输入索引。

In [ ]:
class SimplePointerAttention(nn.Module):
    """
    简化的指针注意力机制。
    
    公式：attention(query, keys) = softmax(v^T tanh(W_q * query + W_k * keys))
    """
    
    def __init__(self, hidden_dim=32):
        super().__init__()
        
        self.W_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, query, keys, mask=None):
        """
        参数：
            query: [batch, hidden_dim]——要查找的内容
            keys: [batch, seq_len, hidden_dim]——搜索的范围
            mask: [batch, seq_len]——1 表示要屏蔽的位置
        返回：
            attention_weights: [batch, seq_len]
        """
        # 变换查询与键
        query_proj = self.W_query(query).unsqueeze(1)  # [batch, 1, hidden_dim]
        keys_proj = self.W_key(keys)                    # [batch, seq_len, hidden_dim]
        
        # 加性注意力
        scores = self.v(torch.tanh(query_proj + keys_proj))  # [batch, seq_len, 1]
        scores = scores.squeeze(-1)  # [batch, seq_len]
        
        # 应用掩码（将被屏蔽位置设为 -inf）
        if mask is not None:
            scores = scores.masked_fill(mask.bool(), float('-inf'))
        
        # 用 softmax 得到概率
        attention_weights = torch.softmax(scores, dim=-1)
        
        return attention_weights

# 测试实现
attention = SimplePointerAttention(hidden_dim=32)

query = torch.randn(1, 32)
keys = torch.randn(1, 5, 32)

weights = attention(query, keys)

print('注意力权重：', weights[0].tolist())
print(f"总和：{weights.sum():.4f}（应为 1.0）")
print('指针注意力已计算出各输入位置的注意力分布。')

### 注意力可视化

下面的柱状图展示给定查询与一组输入项时的注意力权重。

In [ ]:
def visualize_attention_example():
    # 创建 5 个数值不同的输入项
    items = torch.tensor([[0.2, 0.8, 0.1, 0.9, 0.5]])
    
    # 简单嵌入
    embed = nn.Linear(1, 32)
    keys = embed(items.unsqueeze(-1))
    
    # 查询“较大数值”（此处查询是随机生成的，不保证学会该语义）
    query = torch.randn(1, 32)
    
    # 计算注意力
    attention_layer = SimplePointerAttention(32)
    weights = attention_layer(query, keys)
    
    # 可视化
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
    
    # 输入数值
    ax1.bar(range(5), items[0].numpy(), color='skyblue', alpha=0.7)
    ax1.set_title('输入数值', fontsize=14, fontweight='bold')
    ax1.set_xlabel('位置')
    ax1.set_ylabel('数值')
    ax1.grid(True, alpha=0.3)
    
    # 注意力权重
    ax2.bar(range(5), weights[0].detach().numpy(), color='coral', alpha=0.7)
    ax2.set_title('注意力权重（指向哪里）', fontsize=14, fontweight='bold')
    ax2.set_xlabel('位置')
    ax2.set_ylabel('概率')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 显示注意力最高的位置
    best_pos = torch.argmax(weights[0]).item()
    print(f"模型指向位置 {best_pos}（数值：{items[0, best_pos]:.2f})")

visualize_attention_example()

## 第 2 部分：用指针网络排序

我们可以通过数字排序任务测试指针网络。模型需要学习升序排列的“选择规则”。

- **输入：**`[0.7, 0.2, 0.9, 0.1, 0.5]`（无序集合）
- **输出：**`[3, 1, 4, 0, 2]`（按排序顺序排列的索引）

In [ ]:
class PointerNetwork(nn.Module):
    """完整的指针网络教学实现。"""
    
    def __init__(self, input_dim=1, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # 编码器：处理输入集合
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        
        # 解码器：生成输出序列
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        
        # 指针注意力
        self.attention = SimplePointerAttention(hidden_dim)
    
    def forward(self, inputs):
        """
        参数：
            inputs: [batch, seq_len]——要排序的数字
        返回：
            all_pointers: [batch, seq_len, seq_len]——注意力分布
        """
        batch_size, seq_len = inputs.shape
        inputs = inputs.unsqueeze(-1)  # [batch, seq_len, 1]
        
        # 编码
        encoder_outputs, (h, c) = self.encoder(inputs)
        
        # 解码
        decoder_state = (h, c)
        decoder_input = torch.zeros(batch_size, 1, self.hidden_dim, device=inputs.device)
        
        all_pointers = []
        
        for t in range(seq_len):
            # 单步解码
            decoder_output, decoder_state = self.decoder(decoder_input, decoder_state)
            
            # 计算要指向的位置
            weights = self.attention(decoder_output.squeeze(1), encoder_outputs)
            all_pointers.append(weights)
            
            # 下一个输入：编码器输出的加权平均
            context = torch.bmm(weights.unsqueeze(1), encoder_outputs)
            decoder_input = context
        
        return torch.stack(all_pointers, dim=1)

# 初始化模型
model = PointerNetwork(input_dim=1, hidden_dim=64)
print(f"模型参数量：{sum(p.numel() for p in model.parameters()):,}")

### 在排序任务上训练

我们用预测指针索引上的标准交叉熵损失训练模型。请注意开头核验注所述：原代码传入的是概率而非未归一化分数（logits），此处保留原实现。

In [ ]:
def generate_sorting_batch(batch_size=32, seq_len=5):
    """生成随机排序问题。"""
    inputs = torch.rand(batch_size, seq_len)
    targets = torch.argsort(inputs, dim=1)
    return inputs, targets

def train_sorting(model, num_epochs=20, batch_size=32, seq_len=5):
    """训练数字排序模型。"""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    losses = []
    accuracies = []
    
    for epoch in range(num_epochs):
        inputs, targets = generate_sorting_batch(batch_size, seq_len)
        
        logits = model(inputs)
        
        logits_flat = logits.view(-1, seq_len)
        targets_flat = targets.view(-1)
        loss = F.cross_entropy(logits_flat, targets_flat)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        predictions = torch.argmax(logits, dim=-1)
        accuracy = (predictions == targets).all(dim=1).float().mean().item()
        
        losses.append(loss.item())
        accuracies.append(accuracy)
        
        if (epoch + 1) % 5 == 0:
            print(f"轮次 {epoch+1}/{num_epochs}——损失：{loss.item():.4f}，准确率：{accuracy:.2%}")
    
    return losses, accuracies

print('正在长度为 5 的排序序列上训练模型……')
losses, accuracies = train_sorting(model, num_epochs=20, seq_len=5)

# 绘制指标（此处实际为训练批次指标，并非独立验证集指标）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses, color='blue')
ax1.set_title('训练损失')
ax1.set_xlabel('轮次')
ax2.plot(accuracies, color='green')
ax2.set_title('序列准确率')
ax2.set_xlabel('轮次')
plt.tight_layout()
plt.show()

### 推理与注意力热力图

在单个序列上测试模型，并将每一步的注意力绘制为热力图。

In [ ]:
def test_and_visualize(model, numbers):
    model.eval()
    inputs = torch.tensor([numbers]).float()
    
    with torch.no_grad():
        logits = model(inputs)
        predictions = torch.argmax(logits, dim=-1)[0]
    
    true_order = torch.argsort(inputs[0])
    
    fig = plt.figure(figsize=(14, 8))
    
    # 1. 输入数字
    ax1 = plt.subplot(3, 1, 1)
    ax1.bar(range(len(numbers)), numbers, color='skyblue', alpha=0.7)
    ax1.set_title('输入数字（未排序）')
    
    # 2. 注意力热力图
    ax2 = plt.subplot(3, 1, 2)
    attention_matrix = logits[0].softmax(dim=-1).numpy()
    im = ax2.imshow(attention_matrix, cmap='YlOrRd', aspect='auto')
    ax2.set_title('指针注意力热力图')
    ax2.set_ylabel('输出步骤')
    ax2.set_xlabel('输入索引')
    plt.colorbar(im, ax=ax2)
    
    # 3. 预测结果
    ax3 = plt.subplot(3, 1, 3)
    sorted_nums = [numbers[i] for i in predictions.tolist()]
    is_correct = torch.equal(predictions, true_order)
    ax3.bar(range(len(sorted_nums)), sorted_nums, color='lightgreen' if is_correct else 'salmon')
    ax3.set_title(f"预测排序结果（是否正确：{is_correct})")
    
    plt.tight_layout()
    plt.show()

test_numbers = [0.7, 0.2, 0.9, 0.1, 0.5]
test_and_visualize(model, test_numbers)

## 第 3 部分：旅行商问题（TSP）

TSP 是组合优化问题：输入是一组城市位置，目标是找出最短巡回路径。这是 NP 难问题，可用于严格考验指针网络的学习能力。

In [ ]:
class TSPPointerNetwork(nn.Module):
    """带索引屏蔽的 TSP 指针网络。"""
    
    def __init__(self, input_dim=2, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.attention = SimplePointerAttention(hidden_dim)
    
    def forward(self, cities):
        batch_size, num_cities, _ = cities.shape
        encoder_outputs, (h, c) = self.encoder(cities)
        
        decoder_state = (h, c)
        decoder_input = torch.zeros(batch_size, 1, self.hidden_dim, device=cities.device)
        mask = torch.zeros(batch_size, num_cities, device=cities.device)
        
        tour = []
        for t in range(num_cities):
            decoder_output, decoder_state = self.decoder(decoder_input, decoder_state)
            weights = self.attention(decoder_output.squeeze(1), encoder_outputs, mask)
            
            _, selected = weights.max(dim=-1)
            tour.append(selected)
            
            # 标记已选城市，防止重复访问
            mask.scatter_(1, selected.unsqueeze(1), 1)
            
            context = torch.bmm(weights.unsqueeze(1), encoder_outputs)
            decoder_input = context
        
        return torch.stack(tour, dim=1)

tsp_model = TSPPointerNetwork(input_dim=2, hidden_dim=64)
print('TSP 模型已初始化')

## 要点回顾

1. **集合不变性：**原仓库称去除编码器的位置编码即可让指针网络将输入视为集合。此 Notebook 实际使用顺序敏感的 LSTM；不带位置编码的自注意力的逐元素输出则是置换等变的，不能与聚合后的置换不变表示混淆。
2. **指针机制：**模型输出输入中的索引，而非固定词表中的词元，因此可处理可变输入规模，以及词表之外的数值。
3. **泛化：**原仓库称指针网络往往能泛化到比训练时更长的序列，因为它们学习排序等算法操作，而非特定模式。这是需要通过具体实验验证的能力，此处的短训练不构成证明。